# Customer Segmentation Using K-means - walkthrough

**Prodigy InfoTech Machine Learning Task 02.**

This notebook explains the whole project step by step and calls the **same
functions the app and the tests call**, so the numbers here are the project's
numbers. Nothing is re-implemented here.

If the artifacts are missing, create them first:

```powershell
python -m src.train
```

In [ ]:
"""Set up: make the project importable, then load it."""
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():  # started from inside notebooks/
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler

from src import config
from src import visualization as viz
from src.clustering import (
    build_cluster_summary,
    centroids_in_original_units,
    evaluate_candidate_ks,
    fit_pipeline,
    select_k,
)
from src.data import clean_dataset, load_dataset, validate_clustering_input
from src.predict import InputValidationError, assign_profile, assign_profiles, load_artifacts

print("Project root :", PROJECT_ROOT)
print("Dataset file :", config.DATA_FILE.name)
print("Features     :", list(config.FEATURE_COLUMNS))
print("Excluded     :", list(config.EXCLUDED_FROM_MODEL))

## 1. The dataset, and its limits

The file is *Mall Customer Segmentation Data* by Vijay Choudhary, published on
Kaggle. Its own description says the data was created for **learning** customer
segmentation concepts, that it is credited to the Udemy *Machine Learning A-Z*
course, and that Spending Score is "something you assign to the customer based
on your defined parameters like customer behavior and purchasing data".

That last sentence is the important one. **Spending Score is an assigned summary
number, not a measured purchase history.** This dataset has one row per
membership customer and no dated transactions, so it cannot tell us what anyone
bought, how often, or when they last bought. Nothing in this notebook or the app
claims to analyse transaction history, purchase frequency or recency, and no RFM
features are computed, because the data to compute them is not here.

The licence shown on the Kaggle page is "Other (specified in description)", so
the raw file is treated as not redistributable. The row-level output this
notebook produces stays local.

In [ ]:
raw = load_dataset()
print("Shape:", raw.shape)
print("Columns:", list(raw.columns))
display(raw.head())

### Cleaning policy

* **Complete cases only.** A row is kept when both predictors are numeric and
  finite, income is at least 0 k$, and the spending score is a whole number from
  1 to 100. Nothing is imputed, because an invented income or score would be an
  invented customer.
* **Duplicate customers are kept.** Two customers with the same income *and* the
  same spending score are two different people. Only records identical in *every*
  column are treated as duplicated records, and those are dropped keeping the
  first occurrence.
* **Traceability.** Every kept row keeps its 1-based position in the source CSV
  plus the dataset's own `CustomerID`.

In [ ]:
dataset = clean_dataset(raw)
validate_clustering_input(dataset)

for key, value in dataset.report.to_dict().items():
    print(f"{key:32s} {value}")

features = dataset.features
print("\nClustering features, in the fixed order the model uses:")
print(list(features.columns))
print(f"Distinct feature pairs available: {dataset.distinct_feature_pairs}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(features[config.INCOME_COLUMN], bins=20, color="#1f77b4", edgecolor="white")
axes[0].set_xlabel(viz.AXIS_X_LABEL)
axes[0].set_ylabel("Customers")
axes[0].set_title("Annual income (k$)")
axes[1].hist(features[config.SPENDING_SCORE_COLUMN], bins=20, color="#ff7f0e", edgecolor="white")
axes[1].set_xlabel(viz.AXIS_Y_LABEL)
axes[1].set_ylabel("Customers")
axes[1].set_title("Spending score (1-100)")
fig.tight_layout()
plt.show()

## 2. Why scaling comes first

K-means minimises the sum of squared **distances**, and squared distance weights
a column by its variance. StandardScaler gives both features comparable variance
so their numerical scales do not disproportionately influence distances. It does
this by subtracting each feature's mean and dividing by its standard deviation.

Whether an unscaled feature would dominate depends on its actual numerical
spread, not on what its unit is called. In this particular file the two standard
deviations happen to be close (about 26 and 26), so scaling is a safeguard here
rather than a correction. On data whose columns are recorded on very different
scales it changes the answer substantially.

`tests/test_clustering.py` covers both sides: `test_scaling_matters_because_units_differ`
builds columns with very different spread and shows the raw fit chasing the
wrong column, while `test_scaling_is_harmless_when_spreads_are_similar` shows
that comparable spread leaves the raw and standardised fits agreeing.

In [ ]:
scaler_preview = StandardScaler().fit(features)
preview = scaler_preview.transform(features)
print("Raw stds                      :", features.std(ddof=0).round(3).to_dict())
print("Standardised means (want ~0)  :", preview.mean(axis=0).round(10))
print("Standardised stds   (want ~1)  :", preview.std(axis=0).round(6))

## 3. Choosing k

Two metrics with two different jobs:

* **Inertia** is the within-cluster sum of squares. It always falls as k rises, so
  it can never pick k on its own - it is only useful for spotting an elbow.
* The **silhouette score** compares how far a customer sits from its own group
  with how far it sits from the nearest other group. Higher means more separated,
  more internally tight groups. It is a *clustering metric*, not a prediction
  accuracy, and it is only defined for k of 2 or more.

Both are measured in the same standardised matrix the K-means model actually
learns from. Candidates that cannot be scored (k above the sample size, or k
above the number of distinct feature pairs) are marked and skipped rather than
causing a crash. The rule is: **highest measured silhouette wins, and an exact
tie goes to the smaller k.**

In [ ]:
candidates = evaluate_candidate_ks(features, k_max=config.K_MAX)
display(candidates)

selected_k = select_k(candidates)
print("Selected k:", selected_k)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
viz.plot_elbow(candidates, selected_k)
viz.plot_silhouette(candidates, selected_k)
plt.show()

### What this choice does and does not mean

* **The selected k is a practical choice inside the tested range 1 to 10.** It is
  not a discovered truth, and it was not picked because a tutorial used the same
  number - it won on the measured silhouette score.
* The elbow curve is supporting evidence only.
* There are **no ground-truth segment labels** in this dataset, so accuracy,
  precision and recall do not exist here. No such number is reported anywhere.
* All of these numbers are **descriptive and measured on the dataset the model
  was fitted on**. This is not held-out supervised validation.
* **Cluster numbers are arbitrary identifiers**, not a ranking of value.

## 4. Fit the final model and look at the groups

The final fit reuses the same `Pipeline` as the candidate sweep, so the scaler
that was measured is the scaler that gets saved.

In [ ]:
pipeline, labels = fit_pipeline(features, selected_k)
centroids = centroids_in_original_units(pipeline)
summary = build_cluster_summary(features, labels, centroids)

print("Cluster counts sum to the cleaned row count:",
      int(summary["customers"].sum()) == dataset.row_count)
display(summary)

In [ ]:
viz.plot_cluster_scatter(features, labels, centroids, selected_k)
plt.show()

In [ ]:
# The aggregate view the app uses: binned counts plus the centroids. No
# individual customer is recoverable from it, which is why it is the version
# that is safe to publish.
footprint = viz.build_footprint(features, labels)
viz.plot_cluster_footprint(footprint, centroids, selected_k)
plt.show()

In [ ]:
viz.plot_cluster_sizes(summary, selected_k)
plt.show()

## 5. Reading the centroids

Each centroid is a point in the same two-number space, reported back in the
original units. The description is generated from the measurement: a centroid is
called *higher* or *lower* than a feature when it sits more than 10% away from
this dataset's mean for that feature.

The wording stays neutral on purpose. It says where a group sits, and nothing
about age, gender, loyalty or motivation, because the model never saw those
columns and the data cannot support such a claim.

In [ ]:
print("Dataset averages used as the reference point:")
for column in config.FEATURE_COLUMNS:
    value = float(features[column].mean())
    print(f"  {config.FEATURE_LABELS[column]:28s} {value:8.2f}   {config.FEATURE_UNITS[column]}")

display(summary[[
    "cluster_id", "customers",
    "centroid_income_k", "centroid_spending_score", "description",
]])

## 6. Assign a new profile to a group

The saved artifacts are loaded from their fixed path and the *fitted* scaler is
reused. No new scaler is fitted and nothing is retrained.

This is the honest framing: it **assigns an existing spending score to a group**.
It does not predict future spending, and the reported distance is a geometric
distance in the standardised feature space - not a confidence score.

In [ ]:
artifacts = load_artifacts()
print("Loaded:", artifacts.model_path.name, "with k =", artifacts.n_clusters)

for income, score in [(82, 91), (88, 15), (25, 80), (26, 21), (60, 50)]:
    result = assign_profile(artifacts, income, score)
    print(
        f"income={income:>3} k$, score={score:>3}  ->  C{result.cluster_id}  "
        f"{result.description}  (centroid "
        f"{result.centroid[config.INCOME_COLUMN]:.1f} k$, "
        f"{result.centroid[config.SPENDING_SCORE_COLUMN]:.1f})"
    )

In [ ]:
# Validation is shared, so the notebook, the app and the CLI agree.
for bad in [(60, 0), (60, 101), (-5, 50), (60, 50.5), (None, 50)]:
    try:
        assign_profile(artifacts, *bad)
    except InputValidationError as exc:
        print(f"{str(bad):16s} rejected: {exc}")

## 7. The saved model behaves like the fitted one

A saved model is only useful if it reproduces the fitted result exactly. The
check below compares the labels from the loaded pipeline with the labels from the
in-memory fit, and confirms the scaler inside the pipeline is the fitted one.

In [ ]:
reloaded_labels = assign_profiles(artifacts, dataset.features)
print("Saved model reproduces the fitted labels:",
      bool(np.array_equal(reloaded_labels, labels)))
print("Saved scaler mean matches the data mean :",
      np.allclose(artifacts.pipeline.named_steps["scaler"].mean_,
                  features.mean().to_numpy()))
print("\nFitted on  :", artifacts.metadata["trained_at_utc"])
print("Dataset hash:", artifacts.metadata["dataset"]["sha256"])
print("Dependencies:", artifacts.metadata["dependencies"])

In [ ]:
# Row-level assignments, written locally and NOT published.
local = dataset.frame.copy()
local["cluster_id"] = labels
local["segment_description"] = [
    summary.loc[summary["cluster_id"] == int(cid), "description"].iloc[0] for cid in labels
]
print("Rows:", len(local), "| columns:", list(local.columns))
display(local.head())

## Summary

* The file holds one row per membership customer and no dated transactions, so
  Spending Score is an assigned summary number rather than a purchase history.
* Two features: `Annual Income (k$)` and `Spending Score (1-100)`.
  `CustomerID`, `Gender` and `Age` were deliberately left out of the model.
* k was chosen by the highest measured silhouette score inside the tested range
  1-10, not hardcoded.
* The groups are positions in a two-number space, not proven customer
  categories, and the metrics are descriptive rather than predictive.

Next: `python -m streamlit run app.py` to explore the same results
interactively.